# Experiment 2.1b-ii: Layerwise Equivariance Violation vs Muon Condition-Number Proxy

**Counterpart script:** `run_experiment.py` in the same directory.

This notebook is the presentation-and-analysis companion to the script. The script is
now the single source of truth for the experiment core: this notebook imports the
sibling module and consumes its `run_experiment()` return value rather than
re-implementing the training loop.

## Truthful scope of the current study

- **Model:** an 8-layer, 32x32 deep linear network trained on synthetic Gaussian data.
- **Violation metric:** a **target-layer weight-space** discrepancy after orthogonal
  conjugation at initialization and Muon retraining.
- **"Advantage" metric:** `kappa_ratio = cond(W_sgd) / cond(W_muon)` at the end of training.
- **Interpretation:** `kappa_ratio > 1` favors Muon on this particular conditioning proxy;
  `kappa_ratio < 1` favors SGD on this proxy.
- **What this notebook does not claim:** this toy correlation is **not** a mechanistic
  proof, **not** a formal significance result, and **not** a complete measure of optimizer
  advantage.

The pair-level question remains intentionally modest: **do layers with larger measured
Muonic equivariance violation also show larger values of this condition-number proxy?**


## Imports and notebook-safe path resolution

This cell resolves the experiment directory robustly whether the notebook is run from
repo root, from the experiment directory itself, or via an external notebook runner.
It then imports the sibling script by file path.


In [ ]:
from pathlib import Path
import importlib.util
import platform
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 4)

REL_EXPERIMENT_DIR = Path(
    'experiments/Tier_2_Symmetry_Reparametrization_Tests/'
    'Exp_2.1_Conjugation_Covariance/'
    '2.1b-ii_Equivariance_Violation_vs_Advantage'
)


def find_experiment_dir():
    cwd = Path.cwd().resolve()
    if (cwd / 'run_experiment.py').exists():
        return cwd
    for base in [cwd, *cwd.parents]:
        candidate = base / REL_EXPERIMENT_DIR
        if (candidate / 'run_experiment.py').exists():
            return candidate.resolve()
    raise FileNotFoundError(
        'Could not locate the 2.1b-ii experiment directory from the current working directory.'
    )


EXPERIMENT_DIR = find_experiment_dir()
SCRIPT_PATH = EXPERIMENT_DIR / 'run_experiment.py'
NOTEBOOK_PATH = EXPERIMENT_DIR / 'run_experiment.ipynb'

spec = importlib.util.spec_from_file_location(
    'exp_2_1b_ii_run_experiment',
    SCRIPT_PATH,
)
experiment_module = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = experiment_module
spec.loader.exec_module(experiment_module)

print(f'Experiment directory: {EXPERIMENT_DIR}')
print(f'Script path:         {SCRIPT_PATH}')
print(f'Notebook path:       {NOTEBOOK_PATH}')
print('Imported sibling script successfully.')


## Reproducibility, runtime expectations, and default configuration

The default setup is still the original toy study: 5 seeds, depth 8, width 32, and
300 optimization steps. Even at this toy scale, the script performs multiple retraining
passes for the conjugation test, so the run is not instantaneous.


In [ ]:
try:
    import scipy
    scipy_version = scipy.__version__
except Exception:
    scipy_version = 'not available'

default_config = experiment_module.get_default_config()
expected_training_runs = default_config['num_seeds'] * (2 + default_config['depth'])

environment_df = pd.DataFrame([
    {'item': 'python', 'value': platform.python_version()},
    {'item': 'numpy', 'value': np.__version__},
    {'item': 'pandas', 'value': pd.__version__},
    {'item': 'matplotlib', 'value': plt.matplotlib.__version__},
    {'item': 'scipy', 'value': scipy_version},
    {'item': 'cwd', 'value': str(Path.cwd().resolve())},
    {'item': 'experiment_dir', 'value': str(EXPERIMENT_DIR)},
    {'item': 'script_path', 'value': str(SCRIPT_PATH)},
])

config_df = pd.DataFrame([
    {'parameter': key, 'value': value}
    for key, value in default_config.items()
])

print('Environment:')
print(environment_df.to_string(index=False))
print()
print('Default configuration:')
print(config_df.to_string(index=False))
print()
print(f'Expected default training runs: {expected_training_runs}')
print('Runtime note: this CPU toy run typically takes tens of seconds, not milliseconds.')

config_df


## Execute the sibling script via `run_experiment()`

This is the only cell that performs the actual experiment. The notebook intentionally
uses the structured results returned by the script so that the pair stays aligned.


In [ ]:
notebook_t0 = time.time()
results = experiment_module.run_experiment(verbose=False)
notebook_elapsed_sec = time.time() - notebook_t0

print(f'Notebook wall-clock runtime: {notebook_elapsed_sec:.2f}s')
print(f"Script-reported runtime:     {results['runtime']['elapsed_sec']:.2f}s")
print(results['verdict']['summary'])
print(results['verdict']['caution'])


## Configuration, runtime, and headline metrics

The outputs below summarize the exact configuration, runtime, analysis framing,
proxy definition, headline correlation statistics, and coarse optimizer-level loss
diagnostics returned by the sibling script. The primary statistic remains the Pearson
correlation across the **8 layerwise mean pairs** (one aggregated point per layer at
default depth). These are descriptive summaries for this toy run, not a claim of
broader statistical robustness.


In [ ]:
runtime_df = pd.DataFrame([
    {
        'elapsed_sec_reported_by_script': results['runtime']['elapsed_sec'],
        'elapsed_sec_measured_by_notebook': notebook_elapsed_sec,
        'n_training_runs_total': results['runtime']['n_training_runs_total'],
        'n_sgd_training_runs': results['runtime']['n_sgd_training_runs'],
        'n_muon_training_runs': results['runtime']['n_muon_training_runs'],
    }
])

analysis_df = pd.DataFrame([
    {'item': 'primary_question', 'value': results['primary_question']},
    {'item': 'scope_note', 'value': results['scope_note']},
    {'item': 'primary_statistic', 'value': results['analysis_notes']['primary_statistic']},
    {'item': 'secondary_statistic', 'value': results['analysis_notes']['secondary_statistic']},
    {'item': 'layer_point_count', 'value': results['analysis_notes']['layer_point_count']},
    {'item': 'layer_point_note', 'value': results['analysis_notes']['layer_point_note']},
])

proxy_df = pd.DataFrame([
    {'item': 'proxy_name', 'value': results['advantage_proxy']['name']},
    {'item': 'proxy_definition', 'value': results['advantage_proxy']['definition']},
    {'item': 'proxy_interpretation', 'value': results['advantage_proxy']['interpretation']},
    {'item': 'proxy_warning', 'value': results['advantage_proxy']['warning']},
])

correlation_df = pd.DataFrame([
    {
        'metric': 'Pearson on layer means',
        'value': results['correlations']['pearson_layer_means'],
        'notes': 'Primary toy statistic used for T1/T3.',
    },
    {
        'metric': 'Spearman on layer means',
        'value': results['correlations']['spearman_layer_means'],
        'notes': 'Rank-based cross-check; p-value is descriptive only in this tiny layer sample.',
    },
    {
        'metric': 'Mean seedwise Pearson',
        'value': results['aggregate_summaries']['seedwise_pearson']['mean'],
        'notes': 'Correlation computed separately within each seed across the 8 layers.',
    },
    {
        'metric': 'Mean seedwise Spearman',
        'value': results['aggregate_summaries']['seedwise_spearman']['mean'],
        'notes': 'Rank-based within-seed cross-check.',
    },
])

loss_df = pd.DataFrame([
    {
        'optimizer': 'SGD',
        'mean_final_loss': results['aggregate_summaries']['sgd_final_loss']['mean'],
        'std_final_loss': results['aggregate_summaries']['sgd_final_loss']['std'],
    },
    {
        'optimizer': 'Muon',
        'mean_final_loss': results['aggregate_summaries']['muon_final_loss']['mean'],
        'std_final_loss': results['aggregate_summaries']['muon_final_loss']['std'],
    },
])

print('Analysis framing from script results:')
print(analysis_df.to_string(index=False))
print()
print('Proxy definition from script results:')
print(proxy_df.to_string(index=False))
print()
print('Runtime summary:')
print(runtime_df.to_string(index=False))
print()
print('Headline correlations:')
print(correlation_df.to_string(index=False))
print()
print('Final-loss diagnostic:')
print(loss_df.to_string(index=False))

analysis_df


## Per-layer summary table

This is the main quantitative table for the pair. Each row is a layer; the values are
aggregated across seeds. The key comparison is between the layerwise mean violation and
layerwise mean `kappa_ratio`.


In [ ]:
layer_df = pd.DataFrame(results['layer_summaries']).sort_values('layer').reset_index(drop=True)
layer_table = layer_df[[
    'layer',
    'violation_mean',
    'violation_std',
    'kappa_ratio_mean',
    'kappa_ratio_std',
    'kappa_sgd_mean',
    'kappa_muon_mean',
    'muon_favored_fraction',
]].copy()
layer_table['muon_favored_pct'] = 100.0 * layer_table.pop('muon_favored_fraction')

print(layer_table.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

n_muon_favored_layers = int((layer_table['kappa_ratio_mean'] > 1.0).sum())
if n_muon_favored_layers == 0:
    print()
    print(
        'Diagnostic: no layer has mean kappa_ratio > 1 in this run, so this proxy does not '
        'show a Muon advantage at any averaged layer.'
    )
else:
    print()
    print(f'Diagnostic: {n_muon_favored_layers} layer(s) have mean kappa_ratio > 1 in this run.')

layer_table


## Seed-level diagnostics

The experiment is aggregated across seeds, but it is still useful to inspect optimizer
losses and within-seed correlations directly. This does **not** fix the small-sample
nature of the study, but it helps show whether the headline result is stable or fragile
across the five toy replications.


In [ ]:
seed_df = pd.DataFrame([
    {
        'seed': seed_result['seed'],
        'sgd_final_loss': seed_result['sgd']['final_loss'],
        'muon_final_loss': seed_result['muon']['final_loss'],
        'loss_gap_sgd_minus_muon': seed_result['sgd']['final_loss'] - seed_result['muon']['final_loss'],
        'pearson_across_layers': seed_result['pearson_across_layers'],
        'spearman_across_layers': seed_result['spearman_across_layers'],
    }
    for seed_result in results['seed_results']
])

print(seed_df.to_string(index=False, float_format=lambda x: f'{x:.4f}'))
seed_df


## Scatter and profile views of the current correlation metric

- **Left:** layerwise mean violation vs layerwise mean `kappa_ratio`, with SEM error bars.
- **Right:** layer-index profiles for the same two quantities, with SD error bars.

The horizontal dashed line at `kappa_ratio = 1` marks parity on the current proxy.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.errorbar(
    layer_df['violation_mean'],
    layer_df['kappa_ratio_mean'],
    xerr=layer_df['violation_sem'],
    yerr=layer_df['kappa_ratio_sem'],
    fmt='o',
    color='tab:blue',
    ecolor='0.6',
    capsize=3,
)
for _, row in layer_df.iterrows():
    ax.annotate(
        f"L{int(row['layer'])}",
        (row['violation_mean'], row['kappa_ratio_mean']),
        textcoords='offset points',
        xytext=(5, 5),
        fontsize=9,
    )
ax.axhline(1.0, color='tab:red', linestyle='--', linewidth=1, label='kappa_ratio = 1')
ax.set_xlabel('Mean equivariance violation')
ax.set_ylabel('Mean kappa ratio = cond(SGD) / cond(Muon)')
ax.set_title(
    'Layer-mean scatter\n'
    f"Pearson = {results['correlations']['pearson_layer_means']:.3f}, "
    f"Spearman = {results['correlations']['spearman_layer_means']:.3f}"
)
ax.legend(loc='best')

ax2 = axes[1]
ax2.errorbar(
    layer_df['layer'],
    layer_df['violation_mean'],
    yerr=layer_df['violation_std'],
    marker='o',
    color='tab:blue',
    label='Violation mean ± SD',
)
ax2.set_xlabel('Layer index')
ax2.set_ylabel('Violation', color='tab:blue')
ax2.tick_params(axis='y', labelcolor='tab:blue')
ax2.set_title('Layer profiles across depth')

ax2b = ax2.twinx()
ax2b.errorbar(
    layer_df['layer'],
    layer_df['kappa_ratio_mean'],
    yerr=layer_df['kappa_ratio_std'],
    marker='s',
    color='tab:orange',
    label='kappa ratio mean ± SD',
)
ax2b.axhline(1.0, color='tab:red', linestyle='--', linewidth=1)
ax2b.set_ylabel('kappa ratio', color='tab:orange')
ax2b.tick_params(axis='y', labelcolor='tab:orange')

lines1, labels1 = ax2.get_legend_handles_labels()
lines2, labels2 = ax2b.get_legend_handles_labels()
ax2b.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

fig.suptitle('Experiment 2.1b-ii: descriptive views from the script return value', y=1.03)
fig.tight_layout()
plt.show()


## Current test / verdict table

The script still keeps the original toy verdict logic (`T1` and `T3`), but the notebook
now presents it with calibrated language. Passing these tests would still be only a
small descriptive signal, not a strong causal or statistical claim.


In [ ]:
tests_df = pd.DataFrame(results['tests']).copy()
tests_df['pass'] = tests_df['pass'].map({True: 'PASS', False: 'FAIL'})

verdict_df = pd.DataFrame([
    {
        'item': 'supports_positive_relationship',
        'value': results['verdict']['supports_positive_relationship'],
    },
    {
        'item': 'layers_with_mean_kappa_ratio_gt_1',
        'value': results['aggregate_summaries']['layers_with_mean_kappa_ratio_gt_1'],
    },
    {
        'item': 'verdict_summary',
        'value': results['verdict']['summary'],
    },
    {
        'item': 'caution',
        'value': results['verdict']['caution'],
    },
])

print('Toy tests:')
print(tests_df.to_string(index=False))
print()
print('Verdict notes:')
print(verdict_df.to_string(index=False))

tests_df


## Calibrated conclusion

This final cell states plainly whether the current default run supports or rejects the
hypothesized positive relationship, while also recording the toy-study limitations that
still remain after this first completion pass.


In [ ]:
pearson = results['correlations']['pearson_layer_means']
spearman = results['correlations']['spearman_layer_means']
loss_sgd = results['aggregate_summaries']['sgd_final_loss']['mean']
loss_muon = results['aggregate_summaries']['muon_final_loss']['mean']
layers_muon_favored = results['aggregate_summaries']['layers_with_mean_kappa_ratio_gt_1']
depth = results['config']['depth']

if results['verdict']['supports_positive_relationship']:
    support_sentence = 'On this default toy run, the hypothesized positive relationship is descriptively supported.'
else:
    support_sentence = (
        'On this default toy run, the hypothesized positive relationship is not supported: '
        'the layer-mean Pearson correlation is non-positive.'
    )

conclusion_lines = [
    support_sentence,
    f'Observed layer-mean correlations: Pearson = {pearson:.3f}, Spearman = {spearman:.3f}.',
    f'Layers with mean kappa_ratio > 1: {layers_muon_favored} / {depth}.',
    f'Average final loss across seeds: SGD = {loss_sgd:.6e}, Muon = {loss_muon:.6e}.',
    'Interpretation: the present pair only studies a small synthetic deep-linear setting and a single condition-number proxy.',
    'Therefore the result should be treated as a descriptive diagnostic, not as mechanistic proof or as a robust statement about Muon overall.',
]

print('\n'.join(f'- {line}' for line in conclusion_lines))
